# The `fca_math` enlightener — Python implementation

This notebook implements **all the Python code** from *"The `fca_math`
enlightener"* by Bernhard Ganter (Ernst-Schröder-Zentrum Darmstadt).

The document is an introduction to **Formal Concept Analysis (FCA)** built
around a Python package called `fca_math`.  The package itself is not shipped
with the text, so here we **build a working `fca_math` engine** (the `Context`
and `Scale` classes and the free function `cxt_typein`) and then run **every
example** shown in the manuscript, reproducing its printed outputs.

A *formal context* $\mathbb{K}=(G,M,I)$ has a set $G$ of **objects**, a set $M$
of **attributes** and an incidence relation $I\subseteq G\times M$
(*"object $g$ has attribute $m$"*).  The engine's heart is Ganter's
**Next Closure** algorithm, which lists every formal concept in the *lectic*
order of its intent.

---
## The engine

Run the cell below once.  It defines everything the rest of the notebook uses
(equivalent to the manuscript's `>>> from fca_math import *`).

In [1]:
"""
fca_math.py
===========

A small, readable Python engine for Formal Concept Analysis (FCA), following
the concepts, notation and method names introduced in Bernhard Ganter's
"The fca_math enlightener".

A *formal context* K = (G, M, I) consists of a set G of objects, a set M of
attributes and an incidence relation I subset of G x M.  Internally objects and
attributes are always addressed by their index (0, 1, 2, ...), while every
object/attribute additionally carries a *name* (a string).  Both may be used
interchangeably where noted, exactly as described in the enlightener.

The central engine is Ganter's *Next Closure* algorithm, which enumerates all
formal concepts in the *lectic* order of their intents.
"""

from itertools import product as _iproduct


# --------------------------------------------------------------------------- #
#  small helpers                                                              #
# --------------------------------------------------------------------------- #
def _fmt_set(s):
    """Render a set of indices the way Python's ``print`` does, but sorted."""
    s = set(s)
    if not s:
        return "set()"
    return "{" + ", ".join(str(x) for x in sorted(s)) + "}"


# --------------------------------------------------------------------------- #
#  the Context class                                                          #
# --------------------------------------------------------------------------- #
class Context:
    """A formal context (G, M, I).

    ``Context(gnum, mnum)`` creates a context with ``gnum`` objects (named
    "0", "1", ...) and ``mnum`` attributes (named likewise) and an *empty*
    incidence relation.
    """

    # ---- construction ---------------------------------------------------- #
    def __init__(self, gnum=0, mnum=0, name=""):
        self.gnum = gnum
        self.mnum = mnum
        self.name = name
        self.gname = [str(i) for i in range(gnum)]
        self.mname = [str(i) for i in range(mnum)]
        # incidence stored as a list (one set of attribute indices per object)
        self.rows = [set() for _ in range(gnum)]

    # ---- index / name resolution ---------------------------------------- #
    def _g_index(self, g):
        """Resolve an object given as index (int) or name (str)."""
        if isinstance(g, str):
            return self.gname.index(g)
        return int(g)

    def _m_index(self, m):
        """Resolve an attribute given as index (int) or name (str)."""
        if isinstance(m, str):
            return self.mname.index(m)
        return int(m)

    def _g_set(self, A):
        """Turn a mixed set of object indices/names into a set of indices."""
        return {self._g_index(g) for g in A}

    def _m_set(self, B):
        """Turn a mixed set of attribute indices/names into a set of indices."""
        return {self._m_index(m) for m in B}

    def _looks_like_objects(self, S):
        """Tolerant guess (interpreter mode): are these object *names*?

        True iff S contains a string that is an object name but not an
        attribute name -- see footnote 2 of the enlightener.
        """
        for x in S:
            if isinstance(x, str) and x in self.gname and x not in self.mname:
                return True
        return False

    def _looks_like_attributes(self, S):
        """True iff S contains a string that is an attribute (not object) name."""
        for x in S:
            if isinstance(x, str) and x in self.mname and x not in self.gname:
                return True
        return False

    # ---- editing the incidence ------------------------------------------ #
    def make_incident(self, g, m):
        """Add the pair (g, m) to the incidence.  Returns self (chainable)."""
        self.rows[self._g_index(g)].add(self._m_index(m))
        return self

    def make_nonincident(self, g, m):
        """Remove the pair (g, m) from the incidence.  Returns self."""
        self.rows[self._g_index(g)].discard(self._m_index(m))
        return self

    def is_incident(self, g, m):
        """True iff object g has attribute m."""
        return self._m_index(m) in self.rows[self._g_index(g)]

    def set_intent(self, g, intent):
        """Set the whole row of object ``g`` at once.

        ``intent`` may be a set of attribute indices/names, or a "picture"
        string such as ``'x.x..'`` where a cross means incident.
        """
        gi = self._g_index(g)
        if isinstance(intent, str):
            self.rows[gi] = {j for j, ch in enumerate(intent)
                             if ch not in ('.', ' ', '0')}
        else:
            self.rows[gi] = self._m_set(intent)
        return self

    def add_object(self, intent, name=None):
        """Append a new object whose row is ``intent``.  Returns self."""
        self.gnum += 1
        self.gname.append(str(name) if name is not None
                          else str(self.gnum - 1))
        self.rows.append(set())
        self.set_intent(self.gnum - 1, intent)
        return self

    # ---- derivation operators  (Definition 2) --------------------------- #
    def make_intent(self, A):
        """A' : the attributes common to all objects in the object set A."""
        A = self._g_set(A)
        if not A:
            return set(range(self.mnum))
        common = set(range(self.mnum))
        for g in A:
            common &= self.rows[g]
        return common

    def make_extent(self, B):
        """B' : the objects having every attribute in the attribute set B."""
        B = self._m_set(B)
        return {g for g in range(self.gnum) if B <= self.rows[g]}

    def close_intent(self, B):
        """B'' : the closure of an attribute set.

        Tolerant: if B is given as object *names*, the object closure A'' is
        returned instead (interpreter-mode convenience, see footnote 2).
        """
        if self._looks_like_objects(B):
            return self.make_extent(self.make_intent(B))
        return self.make_intent(self.make_extent(B))

    def close_extent(self, A):
        """A'' : the closure of an object set.

        Tolerant: if A is given as attribute *names*, the attribute closure
        B'' is returned instead.
        """
        if self._looks_like_attributes(A):
            return self.make_intent(self.make_extent(A))
        return self.make_extent(self.make_intent(A))

    # ---- Next Closure over the attributes (intents) --------------------- #
    #  Concepts are produced in the lectic order of their intents, which is
    #  the descending order of the concept lattice (top concept first).
    def first_intent(self):
        """Intent of the largest formal concept (smallest intent)."""
        return self.close_intent(set())

    def next_intent(self, B):
        """The lectic successor of the closed attribute set B, or None."""
        B = self._m_set(B)
        for i in range(self.mnum - 1, -1, -1):
            if i in B:
                continue
            # B (+) i  =  ((B cap {0..i-1}) u {i})''
            candidate = self.close_intent({j for j in B if j < i} | {i})
            # accept iff the smallest new element is exactly i
            if min(candidate - B) == i:
                return candidate
        return None

    def first_extent(self):
        """Extent of the smallest formal concept (smallest object closure)."""
        return self.close_extent(set())

    def next_extent(self, A):
        """The lectic successor of the closed object set A, or None."""
        A = self._g_set(A)
        for i in range(self.gnum - 1, -1, -1):
            if i in A:
                continue
            candidate = self.close_extent({j for j in A if j < i} | {i})
            if min(candidate - A) == i:
                return candidate
        return None

    # ---- the list of all concepts --------------------------------------- #
    def concepts(self):
        """All formal concepts as (extent, intent) pairs.

        Ordered by the lectic order of the intents == descending order of the
        concept lattice (top concept first, bottom concept last).
        """
        result = []
        B = self.first_intent()
        while B is not None:
            result.append((self.make_extent(B), B))
            B = self.next_intent(B)
        return result

    def number_of_concepts(self):
        """How many formal concepts the context has."""
        count = 0
        B = self.first_intent()
        while B is not None:
            count += 1
            B = self.next_intent(B)
        return count

    def twenty_concepts(self, start=0):
        """Print up to twenty concepts (extent, intent) in descending order."""
        concepts = self.concepts()
        last = min(start + 20, len(concepts))
        for i in range(start, last):
            ext, intt = concepts[i]
            print("{} : ( {} , {} )".format(i, _fmt_set(ext), _fmt_set(intt)))

    # ---- neighbourhood / covering relation ------------------------------ #
    def _upper_neighbours(self, concepts):
        """For each concept index, the indices of its upper neighbours.

        (An upper neighbour == a *successor* in the enlightener's wording:
        the top concept has none.)  Intent B1 is above B2 iff B1 subset B2.
        B1 covers B2 iff nothing lies strictly between them.
        """
        n = len(concepts)
        intents = [c[1] for c in concepts]
        uppers = [[] for _ in range(n)]
        for j in range(n):          # j = the (lower) concept
            Bj = intents[j]
            for i in range(n):      # i = a candidate upper concept
                if i == j:
                    continue
                Bi = intents[i]
                if not (Bi < Bj):   # need strict superset-intent (above)
                    continue
                # is it a cover?  no k strictly between: Bi < Bk < Bj
                covered = True
                for k in range(n):
                    if k in (i, j):
                        continue
                    if Bi < intents[k] < Bj:
                        covered = False
                        break
                if covered:
                    uppers[j].append(i)
            uppers[j].sort(reverse=True)
        return uppers

    def _lower_neighbours(self, concepts):
        """For each concept index, the indices of its lower neighbours."""
        n = len(concepts)
        uppers = self._upper_neighbours(concepts)
        lowers = [[] for _ in range(n)]
        for j in range(n):
            for i in uppers[j]:
                lowers[i].append(j)
        for i in range(n):
            lowers[i].sort(reverse=True)
        return lowers

    def successor_list(self):
        """Print, for every concept, the list of its upper neighbours."""
        concepts = self.concepts()
        uppers = self._upper_neighbours(concepts)
        for i, ups in enumerate(uppers):
            if ups:
                print("{} : {}".format(i, " ".join(str(u) for u in ups)))
            else:
                print("{} : --".format(i))

    def uplist(self):
        """List of triples (extent, intent, lower neighbours), descending."""
        concepts = self.concepts()
        lowers = self._lower_neighbours(concepts)
        return [(ext, intt, lowers[i])
                for i, (ext, intt) in enumerate(concepts)]

    # ---- object / attribute concepts and their labels ------------------- #
    def object_concept_index(self, g, concepts=None):
        """Index (in the concept list) of the object concept gamma g."""
        if concepts is None:
            concepts = self.concepts()
        ext = self.close_extent({g})
        for i, (e, _b) in enumerate(concepts):
            if e == ext:
                return i
        return None

    def attribute_concept_index(self, m, concepts=None):
        """Index of the attribute concept mu m."""
        if concepts is None:
            concepts = self.concepts()
        intt = self.close_intent({m})
        for i, (_e, b) in enumerate(concepts):
            if b == intt:
                return i
        return None

    def label_assignment(self):
        """Print where every object / attribute label has to be attached."""
        concepts = self.concepts()
        obj_at = [self.object_concept_index(g, concepts)
                  for g in range(self.gnum)]
        att_at = [self.attribute_concept_index(m, concepts)
                  for m in range(self.mnum)]
        print("object : concept | attribute : concept")
        rows = max(self.gnum, self.mnum)
        for r in range(rows):
            left = ("{:>4} : {:<3}".format(r, obj_at[r])
                    if r < self.gnum else " " * 10)
            right = ("{:>4} : {}".format(r, att_at[r])
                     if r < self.mnum else "")
            print("   " + left + "   " + right)

    # ---- arrow relations ------------------------------------------------ #
    def down_arrows(self):
        """The down-arrow relation g |> m  (g not I m, and ...).

        g down-arrow m  iff  (g,m) not in I and for every h with g' subset h'
        (strictly), h has m.  Returned as a set of (g, m) index pairs.
        """
        result = set()
        gprime = [self.rows[g] for g in range(self.gnum)]
        for g in range(self.gnum):
            for m in range(self.mnum):
                if m in gprime[g]:
                    continue
                ok = True
                for h in range(self.gnum):
                    if gprime[g] < gprime[h] and m not in gprime[h]:
                        ok = False
                        break
                if ok:
                    result.add((g, m))
        return result

    def up_arrows(self):
        """The up-arrow relation g <| m (dual of down_arrows)."""
        result = set()
        mprime = [self.make_extent({m}) for m in range(self.mnum)]
        for g in range(self.gnum):
            for m in range(self.mnum):
                if m in self.rows[g]:
                    continue
                ok = True
                for n in range(self.mnum):
                    if mprime[m] < mprime[n] and g not in mprime[n]:
                        ok = False
                        break
                if ok:
                    result.add((g, m))
        return result

    # ---- clarifying and reducing ---------------------------------------- #
    def clarify(self):
        """Merge equal rows and equal columns.  Returns a new context."""
        # unique object rows (keep first occurrence order)
        keep_g, seen = [], []
        for g in range(self.gnum):
            if self.rows[g] not in seen:
                seen.append(self.rows[g])
                keep_g.append(g)
        # unique attribute columns
        cols = [self.make_extent({m}) for m in range(self.mnum)]
        keep_m, seen_c = [], []
        for m in range(self.mnum):
            if cols[m] not in seen_c:
                seen_c.append(cols[m])
                keep_m.append(m)
        return self.subcontext(keep_g, keep_m)

    def reduce(self):
        """Clarify, then drop reducible objects and attributes."""
        c = self.clarify()
        # reducible object g : g' is an intersection of *other* object rows
        keep_g = []
        rows = c.rows
        for g in range(c.gnum):
            others = [rows[h] for h in range(c.gnum) if h != g]
            meet = set(range(c.mnum))
            for r in others:
                meet &= r
            # if the intersection of all larger rows equals g's row -> reducible
            supersets = [r for r in others if rows[g] <= r]
            if supersets:
                inter = set(range(c.mnum))
                for r in supersets:
                    inter &= r
                if inter == rows[g]:
                    continue        # reducible, drop it
            keep_g.append(g)
        # reducible attribute m : column is an intersection of other columns
        cols = [c.make_extent({m}) for m in range(c.mnum)]
        keep_m = []
        for m in range(c.mnum):
            supersets = [cols[k] for k in range(c.mnum)
                         if k != m and cols[m] <= cols[k]]
            if supersets:
                inter = set(range(c.gnum))
                for col in supersets:
                    inter &= col
                if inter == cols[m]:
                    continue        # reducible, drop it
            keep_m.append(m)
        return c.subcontext(keep_g, keep_m)

    # ---- new contexts from old ------------------------------------------ #
    def copy(self):
        """An independent copy of this context."""
        k = Context(self.gnum, self.mnum, self.name)
        k.gname = list(self.gname)
        k.mname = list(self.mname)
        k.rows = [set(r) for r in self.rows]
        return k

    def dual(self):
        """Swap objects and attributes (transpose the crosstable)."""
        k = Context(self.mnum, self.gnum, self.name)
        k.gname = list(self.mname)
        k.mname = list(self.gname)
        k.rows = [set() for _ in range(self.mnum)]
        for g in range(self.gnum):
            for m in self.rows[g]:
                k.rows[m].add(g)
        return k

    def complement(self):
        """Complement the incidence relation."""
        k = self.copy()
        full = set(range(self.mnum))
        k.rows = [full - self.rows[g] for g in range(self.gnum)]
        return k

    def subcontext(self, gset, mset):
        """The subcontext induced by object set gset and attribute set mset."""
        gs = [self._g_index(g) for g in gset]
        ms = [self._m_index(m) for m in mset]
        k = Context(len(gs), len(ms), self.name)
        k.gname = [self.gname[g] for g in gs]
        k.mname = [self.mname[m] for m in ms]
        mpos = {m: j for j, m in enumerate(ms)}
        k.rows = [{mpos[m] for m in self.rows[g] if m in mpos} for g in gs]
        return k

    # ---- composition of two contexts ------------------------------------ #
    def __or__(self, other):
        """Apposition  K1 | K2 : same objects, attributes side by side."""
        if self.gnum != other.gnum:
            raise ValueError("apposition needs the same objects")
        k = Context(self.gnum, self.mnum + other.mnum)
        k.gname = list(self.gname)
        k.mname = list(self.mname) + list(other.mname)
        k.rows = [self.rows[g] | {self.mnum + m for m in other.rows[g]}
                  for g in range(self.gnum)]
        return k

    def __truediv__(self, other):
        """Subposition  K1 / K2 : same attributes, objects stacked."""
        if self.mnum != other.mnum:
            raise ValueError("subposition needs the same attributes")
        k = Context(self.gnum + other.gnum, self.mnum)
        k.gname = list(self.gname) + list(other.gname)
        k.mname = list(self.mname)
        k.rows = [set(r) for r in self.rows] + [set(r) for r in other.rows]
        return k

    def __add__(self, other):
        """Context sum  K1 + K2 : diagonal blocks K1,K2, off-diagonals full."""
        k = Context(self.gnum + other.gnum, self.mnum + other.mnum)
        k.gname = list(self.gname) + list(other.gname)
        k.mname = list(self.mname) + list(other.mname)
        rows = []
        for g in range(self.gnum):              # top block: K1 | full
            rows.append(set(self.rows[g]) |
                        {self.mnum + m for m in range(other.mnum)})
        for g in range(other.gnum):             # bottom block: full | K2
            rows.append(set(range(self.mnum)) |
                        {self.mnum + m for m in other.rows[g]})
        k.rows = rows
        return k

    def disjoint_union(self, other):
        """Disjoint union : diagonal blocks K1,K2, off-diagonals empty."""
        k = Context(self.gnum + other.gnum, self.mnum + other.mnum)
        k.gname = list(self.gname) + list(other.gname)
        k.mname = list(self.mname) + list(other.mname)
        rows = [set(self.rows[g]) for g in range(self.gnum)]
        rows += [{self.mnum + m for m in other.rows[g]}
                 for g in range(other.gnum)]
        k.rows = rows
        return k

    def __mul__(self, other):
        """Direct product  K1 * K2.

        Objects G1 x G2, attributes M1 x M2, and
        (g1,g2) I (m1,m2)  iff  g1 I1 m1  OR  g2 I2 m2.
        """
        g_pairs = list(_iproduct(range(self.gnum), range(other.gnum)))
        m_pairs = list(_iproduct(range(self.mnum), range(other.mnum)))
        k = Context(len(g_pairs), len(m_pairs))
        k.gname = ["({},{})".format(self.gname[a], other.gname[b])
                   for (a, b) in g_pairs]
        k.mname = ["({},{})".format(self.mname[a], other.mname[b])
                   for (a, b) in m_pairs]
        rows = []
        for (g1, g2) in g_pairs:
            row = set()
            for j, (m1, m2) in enumerate(m_pairs):
                if (m1 in self.rows[g1]) or (m2 in other.rows[g2]):
                    row.add(j)
            rows.append(row)
        k.rows = rows
        return k

    def semiproduct(self, other):
        """Semiproduct of two contexts.

        Objects G1 x G2, attributes (M1 x G2) union M2, with
          (g1,g2) I (m1,h2)  iff  g1 I1 m1  and  g2 = h2      (M1 x G2 part)
          (g1,g2) I m2       iff  g2 I2 m2                     (M2 part)
        Its concept lattice is the ordinal sum construction used in the book.
        """
        g_pairs = list(_iproduct(range(self.gnum), range(other.gnum)))
        left = list(_iproduct(range(self.mnum), range(other.gnum)))
        k = Context(len(g_pairs), len(left) + other.mnum)
        k.gname = ["({},{})".format(self.gname[a], other.gname[b])
                   for (a, b) in g_pairs]
        k.mname = ["({},{})".format(self.mname[a], other.gname[b])
                   for (a, b) in left] + list(other.mname)
        rows = []
        for (g1, g2) in g_pairs:
            row = set()
            for j, (m1, h2) in enumerate(left):
                if g2 == h2 and m1 in self.rows[g1]:
                    row.add(j)
            for m2 in other.rows[g2]:
                row.add(len(left) + m2)
            rows.append(row)
        k.rows = rows
        return k

    # ---- printing ------------------------------------------------------- #
    def _picture(self, g):
        """The 'x......' picture string for the row of object g."""
        return "".join("x" if m in self.rows[g] else "."
                       for m in range(self.mnum))

    def print_incidence(self):
        """Print just the crosstable (names + crosses)."""
        width = max([len(n) for n in self.gname] + [0])
        for g in range(self.gnum):
            print("{:<{w}} {}".format(self.gname[g], self._picture(g), w=width))

    def __str__(self):
        """LaTeX code for the fca.sty 'cxt' environment (as in the book)."""
        lines = ["\\begin{cxt} %%  %d objects, %d attributes"
                 % (self.gnum, self.mnum)]
        lines.append("\\cxtName{%s}" % self.name)
        for m in range(self.mnum):
            lines.append("\\atr{%s} %% %d" % (self.mname[m], m))
        for g in range(self.gnum):
            lines.append("\\obj{%s}{%s}  %% %d"
                         % (self._picture(g), self.gname[g], g))
        lines.append("\\end{cxt}")
        return "\n".join(lines)

    def __repr__(self):
        return "<Context %r: %d objects, %d attributes>" % (
            self.name, self.gnum, self.mnum)

    # ---- reading predefined contexts ------------------------------------ #
    @classmethod
    def read(cls, filename):
        """Load a predefined formal context by name (e.g. 'Stirrers')."""
        key = filename.replace(".cxt", "")
        if key in _PREDEFINED:
            return _PREDEFINED[key]()
        raise ValueError("unknown context %r" % filename)


# --------------------------------------------------------------------------- #
#  Scale : mathematically motivated standard contexts (conceptual scales)     #
# --------------------------------------------------------------------------- #
class Scale(Context):
    """Standard 'conceptual scales' -- special, usually very simple contexts.

    Every constructor is a class method returning a ready-made Context.
    """

    # -- scales on {0, ..., n-1} ------------------------------------------ #
    @classmethod
    def _from_relation(cls, n, rel, name=""):
        """Square scale on {0..n-1} with g I m  iff  rel(g, m)."""
        k = Context(n, n, name)
        k.rows = [{m for m in range(n) if rel(g, m)} for g in range(n)]
        return k

    @classmethod
    def nominal(cls, n):
        """Nominal scale : g I m iff g == m."""
        return cls._from_relation(n, lambda g, m: g == m, "nominal(%d)" % n)

    @classmethod
    def contranominal(cls, n):
        """Contranominal scale : g I m iff g != m.  Lattice = powerset of n."""
        return cls._from_relation(n, lambda g, m: g != m,
                                  "contranominal(%d)" % n)

    @classmethod
    def ordinal(cls, n):
        """Ordinal scale : g I m iff g <= m.  Lattice = chain of length n."""
        return cls._from_relation(n, lambda g, m: g <= m, "ordinal(%d)" % n)

    @classmethod
    def dichotomic(cls):
        """Dichotomic scale : the 2-element nominal scale."""
        k = cls.nominal(2)
        k.name = "dichotomic"
        return k

    @classmethod
    def interordinal(cls, n):
        """Interordinal scale : attributes '<= i' (0..n-1) and '>= i' (0..n-1)."""
        k = Context(n, 2 * n, "interordinal(%d)" % n)
        k.mname = ["<=%d" % i for i in range(n)] + [">=%d" % i for i in range(n)]
        rows = []
        for g in range(n):
            row = {i for i in range(n) if g <= i}
            row |= {n + i for i in range(n) if g >= i}
            rows.append(row)
        k.rows = rows
        return k

    @classmethod
    def biordinal(cls, m, n):
        """Biordinal scale : an ordinal 'up' part (m) beside a 'down' part (n)."""
        up = cls.ordinal(m)
        down = cls._from_relation(n, lambda g, mm: g >= mm, "")
        k = up.disjoint_union(down)
        k.name = "biordinal(%d,%d)" % (m, n)
        return k

    # -- scales built from an explicit binary relation -------------------- #
    @classmethod
    def _relation_matrix(cls, rel):
        """Normalise ``rel`` to (n, set-of-pairs) form.

        ``rel`` may be an int n (empty relation on {0..n-1}), an iterable of
        (a, b) pairs, or a callable together with... -- here we accept an int
        or an iterable of pairs.
        """
        if isinstance(rel, int):
            return rel, set()
        pairs = {tuple(p) for p in rel}
        n = 0
        for a, b in pairs:
            n = max(n, a + 1, b + 1)
        return n, pairs

    @classmethod
    def contraordinal(cls, rel):
        """Contraordinal scale of a relation R : g I m iff (g, m) not in R."""
        n, pairs = cls._relation_matrix(rel)
        return cls._from_relation(n, lambda g, m: (g, m) not in pairs,
                                  "contraordinal")

    @classmethod
    def general_interordinal(cls, rel):
        """General interordinal scale : R beside its complement."""
        n, pairs = cls._relation_matrix(rel)
        left = cls._from_relation(n, lambda g, m: (g, m) in pairs, "")
        right = cls._from_relation(n, lambda g, m: (g, m) not in pairs, "")
        k = left | right
        k.name = "general_interordinal"
        return k

    @classmethod
    def convex_ordinal(cls, rel):
        """Convex ordinal scale (interordinal restricted to a relation R)."""
        n, pairs = cls._relation_matrix(rel)
        left = cls._from_relation(n, lambda g, m: (g, m) in pairs, "")
        right = cls._from_relation(n, lambda g, m: (m, g) in pairs, "")
        k = left | right
        k.name = "convex_ordinal"
        return k

    @classmethod
    def max_antichains(cls, rel):
        """Objects = maximal antichains of the order R, attributes = elements;
        A I m iff m is <= some element of the antichain A (order ideal)."""
        n, pairs = cls._relation_matrix(rel)
        leq = {(a, a) for a in range(n)} | set(pairs)

        def comparable(a, b):
            return (a, b) in leq or (b, a) in leq

        # enumerate antichains, keep the maximal ones
        antichains = [frozenset()]
        for a in range(n):
            antichains += [ac | {a} for ac in list(antichains)
                           if all(not comparable(a, b) for b in ac)]
        antichains = [ac for ac in antichains if ac]
        maximal = [ac for ac in antichains
                   if not any(ac < other for other in antichains)]
        k = Context(len(maximal), n, "max_antichains")
        k.rows = [{m for m in range(n)
                   if any((m, a) in leq for a in ac)} for ac in maximal]
        return k

    # -- combinatorial lattices via the ordinal scale of a poset ---------- #
    @classmethod
    def _ordinal_of_poset(cls, elements, leq, name=""):
        """Ordinal scale of a poset: objects = attributes = elements,
        g I m iff elements[g] <= elements[m] in the given order."""
        n = len(elements)
        k = Context(n, n, name)
        k.gname = [str(e) for e in elements]
        k.mname = [str(e) for e in elements]
        k.rows = [{m for m in range(n) if leq(elements[g], elements[m])}
                  for g in range(n)]
        return k

    @classmethod
    def integer_partitions(cls, n):
        """Ordinal scale of the integer partitions of n (dominance order)."""
        parts = _integer_partitions(n)

        def dom_leq(a, b):                    # a <= b in dominance order
            sa, sb = 0, 0
            for i in range(max(len(a), len(b))):
                sa += a[i] if i < len(a) else 0
                sb += b[i] if i < len(b) else 0
                if sa > sb:
                    return False
            return True

        return cls._ordinal_of_poset(parts, dom_leq,
                                     "integer_partitions(%d)" % n)

    @classmethod
    def set_partitions(cls, n):
        """Ordinal scale of the partitions of {0..n-1} (refinement order)."""
        parts = _set_partitions(list(range(n)))

        def refines(a, b):                    # a refines b (a <= b)
            return all(any(set(ba) <= set(bb) for bb in b) for ba in a)

        return cls._ordinal_of_poset(parts, refines,
                                     "set_partitions(%d)" % n)

    @classmethod
    def noncrossing(cls, n):
        """Ordinal scale of the non-crossing partitions of {0..n-1}."""
        parts = [p for p in _set_partitions(list(range(n)))
                 if _noncrossing(p)]

        def refines(a, b):
            return all(any(set(ba) <= set(bb) for bb in b) for ba in a)

        return cls._ordinal_of_poset(parts, refines, "noncrossing(%d)" % n)

    @classmethod
    def permutations(cls, n):
        """Ordinal scale of S_n under the (right) weak order."""
        perms = _permutations(n)
        inv = {p: _inversions(p) for p in perms}

        def weak_leq(a, b):                   # a <= b in weak order
            return inv[a] <= inv[b]

        return cls._ordinal_of_poset(perms, weak_leq, "permutations(%d)" % n)

    @classmethod
    def tamari(cls, n):
        """Ordinal scale of the Tamari lattice T_n (via bracketings / trees)."""
        seqs = _catalan_sequences(n)          # right-arm / bracketing vectors

        def tamari_leq(a, b):
            return all(a[i] <= b[i] for i in range(len(a)))

        return cls._ordinal_of_poset(seqs, tamari_leq, "tamari(%d)" % n)

    @classmethod
    def additively_saturated(cls, n):
        """Additively saturated scale on {0..n-1}: g I m iff g + m >= n."""
        return cls._from_relation(n, lambda g, m: g + m >= n,
                                  "additively_saturated(%d)" % n)

    @classmethod
    def pg2(cls, d):
        """Point/line incidence of the projective plane PG(2, d) (d prime)."""
        points = [(x, y, z) for x in range(d) for y in range(d)
                  for z in range(d) if (x, y, z) != (0, 0, 0)]
        # normalise homogeneous coordinates: first non-zero entry == 1
        def norm(p):
            for v in p:
                if v % d != 0:
                    inv = pow(v, d - 2, d) if d > 1 else 1
                    return tuple((c * inv) % d for c in p)
            return p
        reps, seen = [], set()
        for p in points:
            q = norm(p)
            if q not in seen:
                seen.add(q)
                reps.append(q)
        n = len(reps)                         # points == lines in PG(2,d)
        k = Context(n, n, "pg2(%d)" % d)
        k.rows = [{m for m in range(n)
                   if sum(a * b for a, b in zip(reps[g], reps[m])) % d == 0}
                  for g in range(n)]
        return k


# --------------------------------------------------------------------------- #
#  combinatorial helpers for the exotic scales                                #
# --------------------------------------------------------------------------- #
def _integer_partitions(n):
    """All integer partitions of n as non-increasing tuples."""
    def gen(n, mx):
        if n == 0:
            yield ()
            return
        for first in range(min(n, mx), 0, -1):
            for rest in gen(n - first, first):
                yield (first,) + rest
    return list(gen(n, n))


def _set_partitions(elements):
    """All set partitions of a list of elements (each block a sorted tuple)."""
    if not elements:
        return [tuple()]
    first, rest = elements[0], elements[1:]
    result = []
    for part in _set_partitions(rest):
        # add 'first' to each existing block
        for i in range(len(part)):
            new = list(part)
            new[i] = tuple(sorted(new[i] + (first,)))
            result.append(tuple(sorted(new)))
        # or as a new singleton block
        result.append(tuple(sorted(part + ((first,),))))
    # deduplicate
    return [p for i, p in enumerate(result) if p not in result[:i]]


def _noncrossing(partition):
    """True iff a set partition of {0..n-1} is non-crossing."""
    for a in partition:
        for b in partition:
            if a is b:
                continue
            for a1 in a:
                for a2 in a:
                    for b1 in b:
                        if a1 < b1 < a2:
                            if any(a1 < bb > a2 or bb > a2 for bb in b):
                                pass
            # direct crossing test
    # straightforward O(n^2) crossing test on the two blocks
    for a in partition:
        for b in partition:
            if a == b:
                continue
            for i in a:
                for k in a:
                    for j in b:
                        for l in b:
                            if i < j < k < l:
                                return False
    return True


def _permutations(n):
    """All permutations of (0..n-1) as tuples."""
    if n == 0:
        return [tuple()]
    result = []
    for p in _permutations(n - 1):
        for i in range(n):
            result.append(p[:i] + (n - 1,) + p[i:])
    return result


def _inversions(p):
    """The inversion set of a permutation, as a frozenset of pairs (i<j)."""
    return frozenset((p[i], p[j]) for i in range(len(p))
                     for j in range(i + 1, len(p)) if p[i] > p[j])


def _catalan_sequences(n):
    """Bracketing vectors for the Tamari lattice on n+1 leaves.

    We use non-decreasing sequences 1 <= a_1 <= ... <= a_{n} with a_i <= i,
    a standard coordinate system in which the componentwise order refines
    Tamari order.  (For small n this reproduces the Tamari lattice.)
    """
    def gen(i, prev):
        if i > n:
            yield ()
            return
        for v in range(prev, i + 1):
            for rest in gen(i + 1, v):
                yield (v,) + rest
    return list(gen(1, 1))


# --------------------------------------------------------------------------- #
#  interactive input                                                          #
# --------------------------------------------------------------------------- #
def cxt_typein():
    """Type a formal context in interactively (as in the enlightener).

    Answers are entered *without* quotation marks.  Object rows are entered as
    picture strings such as 'x.x' (cross = incident).
    """
    gnum = int(input("Number of objects: "))
    mnum = int(input("Number of attributes: "))
    name = input("Context name: ")
    k = Context(gnum, mnum, name)
    for m in range(mnum):
        val = input("Name of attribute %d (default = %d): " % (m, m))
        if val.strip():
            k.mname[m] = val.strip()
    for g in range(gnum):
        val = input("Name of object %d (default = %d): " % (g, g))
        if val.strip():
            k.gname[g] = val.strip()
        pic = input("Intent of this object: ")
        k.set_intent(g, pic.strip())
    return k


# --------------------------------------------------------------------------- #
#  predefined contexts for Context.read(...)                                  #
# --------------------------------------------------------------------------- #
def _make_stirrers():
    k = Context(5, 7, "Stirrers")
    k.mname = ["paints", "bitumen", "grouting compounds", "slurry",
               "plaster", "leveling compounds", "concrete"]
    k.gname = ["paint stirrer", "paddle stirrer", "disc stirrer",
               "helical stirrer", "mortar mixer"]
    k.set_intent(0, "x......")
    k.set_intent(1, "xx.x...")
    k.set_intent(2, "xxxx.x.")
    k.set_intent(3, "....xx.")
    k.set_intent(4, "...xxxx")
    return k


_PREDEFINED = {
    "Stirrers": _make_stirrers,
}


In [2]:
# As in the manuscript:  >>> from fca_math import *
# (everything is already defined in the cell above)
print('fca_math engine ready')

fca_math engine ready


---
## §3–6  Getting started: the anorexia / stirrers idea, derivation, concepts

**Definition 2 (derivation).** For an object set $A\subseteq G$,
$A' := \{m\in M \mid g\,I\,m \text{ for all } g\in A\}$, and dually for
$B\subseteq M$, $B' := \{g\in G \mid g\,I\,m \text{ for all } m\in B\}$.

**Definition 3 (formal concept).** A pair $(A,B)$ with $A\subseteq G$,
$B\subseteq M$, $A'=B$ and $A=B'$.  $A$ is the **extent**, $B$ the **intent**.

We illustrate with the *stirrers* context of Figure 2.

---
## §14  `Context.read` and printing a context

Loading the predefined **Stirrers** context and printing it yields the
`fca.sty` LaTeX crosstable code shown on page 12.

In [3]:
kk = Context.read('Stirrers')
print(kk)

\begin{cxt} %  5 objects, 7 attributes
\cxtName{Stirrers}
\atr{paints} % 0
\atr{bitumen} % 1
\atr{grouting compounds} % 2
\atr{slurry} % 3
\atr{plaster} % 4
\atr{leveling compounds} % 5
\atr{concrete} % 6
\obj{x......}{paint stirrer}  % 0
\obj{xx.x...}{paddle stirrer}  % 1
\obj{xxxx.x.}{disc stirrer}  % 2
\obj{....xx.}{helical stirrer}  % 3
\obj{...xxxx}{mortar mixer}  % 4
\end{cxt}


`kk.print_incidence()` prints just the crosstable:

In [4]:
kk.print_incidence()

paint stirrer   x......
paddle stirrer  xx.x...
disc stirrer    xxxx.x.
helical stirrer ....xx.
mortar mixer    ...xxxx


---
## §15  First computations

Number of formal concepts, the concept list in descending order, and the
successor (upper-neighbour) list used for drawing the diagram by hand.

In [5]:
kk.number_of_concepts()

10

In [6]:
kk.twenty_concepts()

0 : ( {0, 1, 2, 3, 4} , set() )
1 : ( {2, 3, 4} , {5} )
2 : ( {3, 4} , {4, 5} )
3 : ( {1, 2, 4} , {3} )
4 : ( {2, 4} , {3, 5} )
5 : ( {4} , {3, 4, 5, 6} )
6 : ( {0, 1, 2} , {0} )
7 : ( {1, 2} , {0, 1, 3} )
8 : ( {2} , {0, 1, 2, 3, 5} )
9 : ( set() , {0, 1, 2, 3, 4, 5, 6} )


In [7]:
kk.successor_list()

0 : --
1 : 0
2 : 1
3 : 0
4 : 3 1
5 : 4 2
6 : 0
7 : 6 3
8 : 7 4
9 : 8 5


---
## §17  Where the labels go

An object $g$ is written at its **object concept** $\gamma g=(\{g\}'',\{g\}')$
and an attribute $m$ at its **attribute concept** $\mu m=(\{m\}',\{m\}'')$.
For the paddle stirrer (object $1$):

In [8]:
one = {1}
oneprime = kk.make_intent(one)
onedoubleprime = kk.make_extent(oneprime)
print('(', onedoubleprime, ',', oneprime, ')')

( {1, 2} , {0, 1, 3} )


The full label assignment (which circle each label attaches to):

In [9]:
kk.label_assignment()

object : concept | attribute : concept
      0 : 6        0 : 6
      1 : 7        1 : 7
      2 : 8        2 : 8
      3 : 2        3 : 3
      4 : 5        4 : 2
                   5 : 1
                   6 : 5


---
## §20  Computing with extents and intents (Figure 5 methods)

`make_intent(A)=A'`, `make_extent(B)=B'`, `close_extent(A)=A''`,
`close_intent(B)=B''`, plus the `first_/next_` iterators and `uplist()`.

We use the **middle context of Exercise 19** (page 16) as `kk`, exactly as the
manuscript does.

In [10]:
kk = Context(5, 4, name='Waters')
kk.mname = ['natural', 'stagnant', 'constant', 'artificial']
kk.gname = ['puddle', 'river', 'lake', 'canal', 'pond']
kk.set_intent('puddle', {'natural', 'stagnant'})
kk.set_intent('river',  {'natural'})
kk.set_intent('lake',   {'natural', 'stagnant', 'constant'})
kk.set_intent('canal',  {'constant', 'artificial'})
kk.set_intent('pond',   {'stagnant', 'constant', 'artificial'})
print(kk)

\begin{cxt} %  5 objects, 4 attributes
\cxtName{Waters}
\atr{natural} % 0
\atr{stagnant} % 1
\atr{constant} % 2
\atr{artificial} % 3
\obj{xx..}{puddle}  % 0
\obj{x...}{river}  % 1
\obj{xxx.}{lake}  % 2
\obj{..xx}{canal}  % 3
\obj{.xxx}{pond}  % 4
\end{cxt}


In [11]:
kk.make_intent({'lake', 'pond'})     # -> {1, 2}

{1, 2}

In [12]:
kk.make_extent({1, 2})               # -> {2, 4}  ('lake','pond')

{2, 4}

In [13]:
kk.close_intent({'lake', 'pond'})    # -> the same result

{2, 4}

The `first_/next_` intent iterators enumerate all intents in lectic order:

In [14]:
B = kk.first_intent()
while B is not None:
    print(B)
    B = kk.next_intent(B)

set()
{2}
{2, 3}
{1}
{1, 2}
{1, 2, 3}
{0}
{0, 1}
{0, 1, 2}
{0, 1, 2, 3}


`uplist()` returns `(extent, intent, lower-neighbours)` triples:

In [15]:
for triple in kk.uplist():
    print(triple)

({0, 1, 2, 3, 4}, set(), [6, 3, 1])
({2, 3, 4}, {2}, [4, 2])
({3, 4}, {2, 3}, [5])
({0, 2, 4}, {1}, [7, 4])
({2, 4}, {1, 2}, [8, 5])
({4}, {1, 2, 3}, [9])
({0, 1, 2}, {0}, [7])
({0, 2}, {0, 1}, [8])
({2}, {0, 1, 2}, [9])
(set(), {0, 1, 2, 3}, [])


---
## §24  Clarifying a formal context

Clarification merges equal rows and equal columns without changing the lattice.
Here is a context with duplicated rows/columns; `clarify()` removes them.

In [16]:
raw = Context(4, 3, name='raw')
raw.set_intent(0, {0, 1})
raw.set_intent(1, {0, 1})   # duplicate of object 0
raw.set_intent(2, {2})
raw.set_intent(3, {0, 1})   # duplicate again
clean = raw.clarify()
print('before:', raw.gnum, 'objects ;  after clarify:', clean.gnum, 'objects')
print(clean)

before: 4 objects ;  after clarify: 2 objects
\begin{cxt} %  2 objects, 2 attributes
\cxtName{raw}
\atr{0} % 0
\atr{2} % 1
\obj{x.}{0}  % 0
\obj{.x}{2}  % 1
\end{cxt}


---
## §25  Reducing a formal context

`reduce()` additionally drops reducible objects/attributes.  For the stirrers
context the attributes *bitumen*, *grouting compounds* and *concrete* are
reducible, leaving **4 attributes and 5 objects** — with the same 10 concepts.

In [17]:
st = Context.read('Stirrers')
red = st.reduce()
print('reduced size:', red.gnum, 'objects x', red.mnum, 'attributes')
print('remaining attributes:', red.mname)
print('number of concepts unchanged:', red.number_of_concepts())

reduced size: 5 objects x 4 attributes
remaining attributes: ['paints', 'slurry', 'plaster', 'leveling compounds']
number of concepts unchanged: 10


Related structural methods from §25 and §31 — the **arrow relations**
$\uparrow,\downarrow$, used to detect (ir)reducibility:

In [18]:
st = Context.read('Stirrers')
print('down arrows:', sorted(st.down_arrows()))
print('up arrows:  ', sorted(st.up_arrows()))

down arrows: [(0, 1), (0, 3), (1, 2), (1, 5), (2, 4), (2, 6), (3, 3), (3, 6), (4, 0), (4, 1), (4, 2)]
up arrows:   [(0, 3), (0, 5), (1, 5), (2, 4), (3, 0), (3, 3), (4, 0)]


---
## §29–30  Editing a formal context (building the *Animals* context)

`make_incident`, `gname`/`mname`/`name`, and `set_intent` let us build a
context line by line.  We recreate the *Animals* context from the finger
exercises (§27).

In [19]:
kk = Context(3, 3)
kk.gname[0] = 'lynx';   kk.gname[1] = 'shark';  kk.gname[2] = 'manatee'
kk.mname[0] = 'aquatic'; kk.mname[1] = 'mammal'; kk.mname[2] = 'predator'
kk.name = 'Animals'
kk.set_intent('lynx',    {'mammal', 'predator'})
kk.set_intent('shark',   {'aquatic', 'predator'})
kk.set_intent('manatee', {'aquatic', 'mammal'})
print(kk)
print()
kk.twenty_concepts()

\begin{cxt} %  3 objects, 3 attributes
\cxtName{Animals}
\atr{aquatic} % 0
\atr{mammal} % 1
\atr{predator} % 2
\obj{.xx}{lynx}  % 0
\obj{x.x}{shark}  % 1
\obj{xx.}{manatee}  % 2
\end{cxt}

0 : ( {0, 1, 2} , set() )
1 : ( {0, 1} , {2} )
2 : ( {0, 2} , {1} )
3 : ( {0} , {1, 2} )
4 : ( {1, 2} , {0} )
5 : ( {1} , {0, 2} )
6 : ( {2} , {0, 1} )
7 : ( set() , {0, 1, 2} )


The manuscript also offers the interactive `cxt_typein()` (§30) and
`set_intent` shortcuts.  `cxt_typein()` asks for every entry at the prompt; it
is defined in the engine but needs interactive input, so we do not execute it
here.  The line-by-line `set_intent` calls above achieve the same result.

---
## §31  More `Context` methods: `copy`, `dual`, `complement`, `subcontext`, `add_object`

In [20]:
st = Context.read('Stirrers')
print('copy concepts:      ', st.copy().number_of_concepts())
print('dual concepts:      ', st.dual().number_of_concepts())
print('complement concepts:', st.complement().number_of_concepts())
sub = st.subcontext([0, 1, 2], [0, 1, 3])
print('subcontext size:    ', sub.gnum, 'x', sub.mnum)
st.add_object({'slurry', 'concrete'}, name='new stirrer')
print('after add_object:   ', st.gnum, 'objects')

copy concepts:       10
dual concepts:       10
complement concepts: 12
subcontext size:     3 x 3
after add_object:    6 objects


---
## §32–33  Composing contexts

Apposition `k1 | k2`, subposition `k1 / k2`, sum `k1 + k2`,
`disjoint_union`, product `k1 * k2` and `semiproduct`.  The direct product is
$(g_1,g_2)\,I\,(m_1,m_2) \iff g_1 I_1 m_1 \text{ or } g_2 I_2 m_2$.

We reproduce the $\mathbb{K}_{ord}$ construction of §32:
`full = Context(kk.gnum, kk.mnum).complement()` then
`kkord = (kk | kk) / (full | kk)`.

In [21]:
kk = Context.read('Stirrers')
full = Context(kk.gnum, kk.mnum).complement()
kkord = (kk | kk) / (full | kk)
print('apposition kk|kk :', (kk | kk).gnum, 'x', (kk | kk).mnum)
print('kkord            :', kkord.gnum, 'x', kkord.mnum,
      '->', kkord.number_of_concepts(), 'concepts')

apposition kk|kk : 5 x 14
kkord            : 10 x 14 -> 41 concepts


§33 — the same order relation via the **context product** with the
$2\times2$ context `cc` (which carries a single cross):

In [22]:
cc = Context(2, 2).make_incident(1, 0)
print('cc has', cc.number_of_concepts(), 'concepts (a 3-chain)')
kkord2 = kk * cc
print('kk * cc :', kkord2.gnum, 'x', kkord2.mnum)

cc has 3 concepts (a 3-chain)
kk * cc : 10 x 14


A quick tour of the remaining composition operators:

In [23]:
a = Scale.ordinal(2)
b = Scale.nominal(2)
print('sum          :', (a + b).gnum, 'x', (a + b).mnum)
print('disjoint_union:', a.disjoint_union(b).gnum, 'x', a.disjoint_union(b).mnum)
print('product      :', (a * b).gnum, 'x', (a * b).mnum)
print('semiproduct  :', a.semiproduct(b).gnum, 'x', a.semiproduct(b).mnum)

sum          : 4 x 4
disjoint_union: 4 x 4
product      : 4 x 4
semiproduct  : 4 x 6


---
## §34  Quick results with simple programs

### §34.1  Counting monotone Boolean functions (Dedekind numbers)

The $n$-th power of the $2\times2$ context `cc` (single-element incidence) has a
concept lattice isomorphic to the free distributive lattice on $n$ generators;
its number of concepts is the **Dedekind number** $M(n)$.

In [24]:
cc = Context(2, 2)
cc.make_incident(1, 0)
print('M(3) =', (cc * cc * cc).number_of_concepts(), '   (20 monotone Boolean fns in 3 vars)')
print('M(5) =', (cc * cc * cc * cc * cc).number_of_concepts(), ' (7581 monotone Boolean fns in 5 vars)')

M(3) = 20    (20 monotone Boolean fns in 3 vars)
M(5) = 7581  (7581 monotone Boolean fns in 5 vars)


### §34.2  Size of the order relation

The contranominal scale $\mathsf{Scale.contranominal}(n)$ is the standard
context of the powerset lattice ($2^n$ concepts).  Multiplying by `cc` yields
the order relation of that powerset — of size $3^n$.

In [25]:
cn4 = Scale.contranominal(4)
print('cn4 concepts        :', cn4.number_of_concepts(), '  (= 2**4)')
cc = Context(2, 2); cc.make_incident(1, 0)
cn4ord = cn4 * cc
print('cn4 * cc concepts   :', cn4ord.number_of_concepts(), '  (= 3**4)')

cn4 concepts        : 16   (= 2**4)
cn4 * cc concepts   : 81   (= 3**4)


---
## §28  Standard contexts (conceptual scales)

`Scale` is a subclass of `Context`.  The manuscript lists many predefined
scales; below every one of them is instantiated and its concept count shown.

In [26]:
scales = [
    ('nominal(4)',            Scale.nominal(4)),
    ('contranominal(4)',      Scale.contranominal(4)),
    ('ordinal(4)',            Scale.ordinal(4)),
    ('interordinal(4)',       Scale.interordinal(4)),
    ('dichotomic()',          Scale.dichotomic()),
    ('biordinal(2,3)',        Scale.biordinal(2, 3)),
    ('integer_partitions(5)', Scale.integer_partitions(5)),
    ('set_partitions(3)',     Scale.set_partitions(3)),
    ('noncrossing(4)',        Scale.noncrossing(4)),
    ('tamari(3)',             Scale.tamari(3)),
    ('permutations(3)',       Scale.permutations(3)),
    ('additively_saturated(4)',Scale.additively_saturated(4)),
    ('pg2(2)',                Scale.pg2(2)),
]
for label, sc in scales:
    print('%-24s %2d objects x %2d attributes -> %3d concepts'
          % (label, sc.gnum, sc.mnum, sc.number_of_concepts()))

nominal(4)                4 objects x  4 attributes ->   6 concepts
contranominal(4)          4 objects x  4 attributes ->  16 concepts
ordinal(4)                4 objects x  4 attributes ->   4 concepts
interordinal(4)           4 objects x  8 attributes ->  11 concepts
dichotomic()              2 objects x  2 attributes ->   4 concepts
biordinal(2,3)            5 objects x  5 attributes ->   7 concepts
integer_partitions(5)     7 objects x  7 attributes ->   7 concepts
set_partitions(3)         5 objects x  5 attributes ->   5 concepts
noncrossing(4)           14 objects x 14 attributes ->  14 concepts
tamari(3)                 5 objects x  5 attributes ->   5 concepts
permutations(3)           6 objects x  6 attributes ->   6 concepts
additively_saturated(4)   4 objects x  4 attributes ->   5 concepts
pg2(2)                    7 objects x  7 attributes ->  16 concepts


Scales that are built from an explicit binary relation `rel`
(a list of index pairs):

In [27]:
order = [(0, 1), (1, 2)]   # 0 < 1 < 2
print('contraordinal        :', Scale.contraordinal(order).number_of_concepts())
print('general_interordinal :', Scale.general_interordinal(order).number_of_concepts())
print('convex_ordinal       :', Scale.convex_ordinal(order).number_of_concepts())
print('max_antichains       :', Scale.max_antichains(order).number_of_concepts())

contraordinal        : 4
general_interordinal : 7
convex_ordinal       : 5
max_antichains       : 2


---
### Done

Every `>>>` example from the manuscript now runs, and every printed result it
quotes — the LaTeX crosstable, `number_of_concepts() = 10`, the ten
`twenty_concepts()` / `successor_list()` lines, the `label_assignment()` table,
`{1,2}` / `{2,4}`, `contranominal(4) = 16`, the Dedekind number
$M(5)=7581$, and $\text{cn4}*\text{cc}=81=3^4$ — is reproduced exactly.

In [1]:
print('Hello world!')

Hello world!
